# Retail Data Wrangling and Analytics Using PySpark


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.table("lgs_databricks.lgs_retail.retail")
display(df)
df.printSchema()
df.count()

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
558461,20725,LUNCH BAG RED RETROSPOT,2,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558461,21932,SCANDINAVIAN PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558475,21530,DAIRY MAID TOASTRACK,1,2011-06-29T15:58:00.000Z,3.29,null,United Kingdom
558461,21933,PINK VINTAGE PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558462,21533,RETROSPOT LARGE MILK JUG,2,2011-06-29T15:11:00.000Z,4.95,13982,United Kingdom
558462,20724,RED RETROSPOT CHARLOTTE BAG,20,2011-06-29T15:11:00.000Z,0.85,13982,United Kingdom
558462,20718,RED RETROSPOT SHOPPER BAG,10,2011-06-29T15:11:00.000Z,1.25,13982,United Kingdom
558462,85099B,JUMBO BAG RED RETROSPOT,10,2011-06-29T15:11:00.000Z,2.08,13982,United Kingdom
558462,82482,WOODEN PICTURE FRAME WHITE FINISH,6,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom
558462,21531,RED RETROSPOT SUGAR JAM BOWL,4,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom


root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- country: string (nullable = true)



1067371

# Section 0: Data understanding

In [0]:
# Section 0 — Data Understanding & Validation
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load table
df = spark.table("lgs_databricks.lgs_retail.retail")

# Quick peek: schema + sample + row count
df.printSchema()
display(df.limit(10))
print("rows =", df.count())

# Coverage: date range
print("date range:")
df.select(
    F.min("invoice_date").alias("min_invoice_date"),
    F.max("invoice_date").alias("max_invoice_date")
).show(truncate=False)

# Null profile
print("null profile:")
nulls = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df.columns
])
display(nulls)

# Basic uniqueness checks
print("uniqueness checks:")
df.select(
    F.count("*").alias("rows"),
    F.countDistinct("invoice_no").alias("distinct_invoice_no"),
    F.countDistinct("customer_id").alias("distinct_customer_id"),
    F.countDistinct("stock_code").alias("distinct_stock_code"),
    F.countDistinct("country").alias("distinct_country")
).show()

# Cancellations: how many + what % of rows
print("cancellations:")
total_rows = df.count()
cancel_rows = df.filter(F.col("invoice_no").startswith("C")).count()
print("total_rows =", total_rows)
print("cancel_rows =", cancel_rows)
print("cancel_pct  =", round(cancel_rows * 100.0 / total_rows, 4), "%")

# Quantity & price sanity
print("quantity & price sanity:")
df.select(
    F.min("quantity").alias("min_qty"),
    F.max("quantity").alias("max_qty"),
    F.min("unit_price").alias("min_unit_price"),
    F.max("unit_price").alias("max_unit_price")
).show()

# Count invalid rows
print("invalid records")
df.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("qty_le_0"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("price_le_0")
).show()

# Country distribution (top 10)
print("country distribution:")
top_countries = (df.groupBy("country")
                   .count()
                   .orderBy(F.desc("count"))
                   .limit(10))
display(top_countries)

root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- country: string (nullable = true)



invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
558461,20725,LUNCH BAG RED RETROSPOT,2,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558461,21932,SCANDINAVIAN PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558475,21530,DAIRY MAID TOASTRACK,1,2011-06-29T15:58:00.000Z,3.29,null,United Kingdom
558461,21933,PINK VINTAGE PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom
558462,21533,RETROSPOT LARGE MILK JUG,2,2011-06-29T15:11:00.000Z,4.95,13982,United Kingdom
558462,20724,RED RETROSPOT CHARLOTTE BAG,20,2011-06-29T15:11:00.000Z,0.85,13982,United Kingdom
558462,20718,RED RETROSPOT SHOPPER BAG,10,2011-06-29T15:11:00.000Z,1.25,13982,United Kingdom
558462,85099B,JUMBO BAG RED RETROSPOT,10,2011-06-29T15:11:00.000Z,2.08,13982,United Kingdom
558462,82482,WOODEN PICTURE FRAME WHITE FINISH,6,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom
558462,21531,RED RETROSPOT SUGAR JAM BOWL,4,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom


rows = 1067371
date range:
+-------------------+-------------------+
|min_invoice_date   |max_invoice_date   |
+-------------------+-------------------+
|2009-12-01 07:45:00|2011-12-09 12:50:00|
+-------------------+-------------------+

null profile:


invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,0,4382,0,0,0,243007,0


uniqueness checks:
+-------+-------------------+--------------------+-------------------+----------------+
|   rows|distinct_invoice_no|distinct_customer_id|distinct_stock_code|distinct_country|
+-------+-------------------+--------------------+-------------------+----------------+
|1067371|              53628|                5942|               5305|              43|
+-------+-------------------+--------------------+-------------------+----------------+

cancellations:
total_rows = 1067371
cancel_rows = 19494
cancel_pct  = 1.8264 %
quantity & price sanity:
+-------+-------+--------------+--------------+
|min_qty|max_qty|min_unit_price|max_unit_price|
+-------+-------+--------------+--------------+
| -80995|  80995|      -53594.4|       38970.0|
+-------+-------+--------------+--------------+

invalid records
+--------+----------+
|qty_le_0|price_le_0|
+--------+----------+
|   22950|      6207|
+--------+----------+

country distribution:


country,count
United Kingdom,981330
EIRE,17866
Germany,17624
France,14330
Netherlands,5140
Spain,3811
Switzerland,3189
Belgium,3123
Portugal,2620
Australia,1913


In [0]:
letter_only_codes = (df
    .filter(F.col("stock_code").rlike("^[A-Z ]+$"))  # Only uppercase letters and spaces
    .select("stock_code", "description")
    .distinct()
    .orderBy("stock_code")
)

display(letter_only_codes)


stock_code,description
ADJUST,Adjustment by Peter on 24/05/2010 1
ADJUST,Adjustment by john on 26/01/2010 16
ADJUST,Adjustment by john on 26/01/2010 17
AMAZONFEE,AMAZON FEE
B,Adjust bad debt
BANK CHARGES,Bank Charges
BANK CHARGES,Bank Charges
CRUK,CRUK Commission
D,Discount
DCGSLBOY,null


In [0]:
# Method 2: Find codes with extreme prices (likely adjustments)
extreme_prices = (df
    .filter((F.col("unit_price") > 1000) | (F.col("unit_price") < -100))
    .groupBy("stock_code", "description")  # Add groupBy before aggregation
    .agg(F.max("unit_price").alias("max_unit_price"))
    .orderBy(F.desc("max_unit_price"))
)

display(extreme_prices)

stock_code,description,max_unit_price
M,Manual,38970.0
BANK CHARGES,Bank Charges,18910.7
AMAZONFEE,AMAZON FEE,17836.5
B,Adjust bad debt,11062.1
POST,POSTAGE,8142.75
ADJUST,Adjustment by john on 26/01/2010 17,5117.03
DOT,DOTCOM POSTAGE,4505.17
D,Discount,1867.86
84016,FLAG OF ST GEORGE CAR FLAG,1157.15
CRUK,CRUK Commission,1100.44


In [0]:
display(df.filter(F.col("stock_code").rlike("DCGSSBOY")))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_cancelled,revenue,year_month
499040,DCGSSBOY,null,-90,2010-02-24T13:16:00.000Z,0.0,null,United Kingdom,false,0.0,201002
512737,DCGSSBOY,update,100,2010-06-17T14:10:00.000Z,0.0,null,United Kingdom,false,0.0,201006
513099,DCGSSBOY,BOYS PARTY BAG,5,2010-06-21T15:13:00.000Z,3.36,null,United Kingdom,false,16.8,201006
513200,DCGSSBOY,BOYS PARTY BAG,7,2010-06-22T16:28:00.000Z,3.36,null,United Kingdom,false,23.52,201006
513574,DCGSSBOY,BOYS PARTY BAG,3,2010-06-25T15:13:00.000Z,3.36,null,United Kingdom,false,10.08,201006
513655,DCGSSBOY,BOYS PARTY BAG,4,2010-06-28T10:02:00.000Z,3.36,null,United Kingdom,false,13.44,201006
516470,DCGSSBOY,BOYS PARTY BAG,1,2010-07-20T15:28:00.000Z,3.36,null,United Kingdom,false,3.36,201007
526405,DCGSSBOY,BOYS PARTY BAG,1,2010-10-11T13:53:00.000Z,3.36,null,United Kingdom,false,3.36,201010
530060,DCGSSBOY,BOYS PARTY BAG,1,2010-11-01T11:35:00.000Z,3.36,null,United Kingdom,false,3.36,201011
530140,DCGSSBOY,BOYS PARTY BAG,4,2010-11-01T16:45:00.000Z,3.36,null,United Kingdom,false,13.44,201011


In [0]:
# Add helper columns for analysis

df = (df
    .withColumn("is_cancelled", F.col("invoice_no").startswith("C"))
    .withColumn("revenue",F.round(F.col("quantity") * F.col("unit_price"), 2))  # Round to 2 decimals
    .withColumn("year_month", F.date_format("invoice_date", "yyyyMM").cast("int"))
)

In [0]:
# Investigation: Are non-product codes standalone or mixed with products?

print("=== Checking if non-product codes appear in multi-item invoices ===\n")

non_product_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'DOT', 'POST', 'ADJUST', 'D', 'CRUK']

for code in non_product_codes:
    # Get invoices containing this code
    invoices_with_code = df.filter(F.col("stock_code") == code).select("invoice_no").distinct()
    
    # For these invoices, count how many line items they have
    invoice_line_counts = (df
        .join(invoices_with_code, "invoice_no")
        .groupBy("invoice_no")
        .agg(F.count("*").alias("line_items"))
    )
    
    # How many are standalone (1 item) vs. multi-item?
    standalone = invoice_line_counts.filter(F.col("line_items") == 1).count()
    multi_item = invoice_line_counts.filter(F.col("line_items") > 1).count()
    total = invoice_line_counts.count()
    
    print(f"{code:15} | Total invoices: {total:5} | Standalone: {standalone:5} | Multi-item: {multi_item:5}")

=== Checking if non-product codes appear in multi-item invoices ===

M               | Total invoices:  1286 | Standalone:   679 | Multi-item:   607
B               | Total invoices:     6 | Standalone:     6 | Multi-item:     0
BANK CHARGES    | Total invoices:    95 | Standalone:    87 | Multi-item:     8
AMAZONFEE       | Total invoices:    36 | Standalone:    29 | Multi-item:     7
DOT             | Total invoices:  1425 | Standalone:     2 | Multi-item:  1423
POST            | Total invoices:  2084 | Standalone:   275 | Multi-item:  1809
ADJUST          | Total invoices:    67 | Standalone:    67 | Multi-item:     0
D               | Total invoices:   151 | Standalone:   130 | Multi-item:    21
CRUK            | Total invoices:    16 | Standalone:    16 | Multi-item:     0


In [0]:
display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_cancelled,revenue,year_month
558461,20725,LUNCH BAG RED RETROSPOT,2,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom,false,3.3,201106
558461,21932,SCANDINAVIAN PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom,false,4.95,201106
558475,21530,DAIRY MAID TOASTRACK,1,2011-06-29T15:58:00.000Z,3.29,null,United Kingdom,false,3.29,201106
558461,21933,PINK VINTAGE PAISLEY PICNIC BAG,3,2011-06-29T15:03:00.000Z,1.65,13263,United Kingdom,false,4.95,201106
558462,21533,RETROSPOT LARGE MILK JUG,2,2011-06-29T15:11:00.000Z,4.95,13982,United Kingdom,false,9.9,201106
558462,20724,RED RETROSPOT CHARLOTTE BAG,20,2011-06-29T15:11:00.000Z,0.85,13982,United Kingdom,false,17.0,201106
558462,20718,RED RETROSPOT SHOPPER BAG,10,2011-06-29T15:11:00.000Z,1.25,13982,United Kingdom,false,12.5,201106
558462,85099B,JUMBO BAG RED RETROSPOT,10,2011-06-29T15:11:00.000Z,2.08,13982,United Kingdom,false,20.8,201106
558462,82482,WOODEN PICTURE FRAME WHITE FINISH,6,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom,false,15.3,201106
558462,21531,RED RETROSPOT SUGAR JAM BOWL,4,2011-06-29T15:11:00.000Z,2.55,13982,United Kingdom,false,10.2,201106


# Total Invoice Amount Distribution

In [0]:

# Section 1: Total Invoice Amount Distribution


# Exclude accounting entries (identified through data exploration)
# These are 80%+ standalone entries, not part of customer transactions
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Calculate invoice amounts (products + shipping, exclude accounting noise)
invoice_amounts = (df
    .filter(~F.col("stock_code").isin(exclude_codes))  # Remove accounting entries
    .filter(~F.col("is_cancelled"))  # Only completed orders
    .groupBy("invoice_no")
    .agg(F.round(F.sum("revenue"), 2).alias("invoice_amount"))
    .filter(F.col("invoice_amount") > 0)  # Positive amounts only
)

print(f"Total invoices: {invoice_amounts.count():,}")

# Full distribution statistics
print("\n=== Full Distribution (with outliers) ===")
stats_full = invoice_amounts.select(
    F.min("invoice_amount").alias("min"),
    F.max("invoice_amount").alias("max"),
    F.mean("invoice_amount").alias("mean"),
    F.expr("percentile_approx(invoice_amount, 0.5)").alias("median")
)
stats_full.show()

# Mode (most frequent invoice amount)
mode_full = (invoice_amounts
    .groupBy("invoice_amount")
    .count()
    .orderBy(F.desc("count"))
    .limit(1)
    .select("invoice_amount")
    .first()[0]
)
print(f"Mode: £{mode_full}")

# Display for visualization
display(invoice_amounts)

# Filter to 85th percentile (remove top 15% outliers)
percentile_85 = invoice_amounts.approxQuantile("invoice_amount", [0.85], 0.01)[0]
print(f"\n85th percentile threshold: £{percentile_85:.2f}")

invoice_amounts_filtered = invoice_amounts.filter(F.col("invoice_amount") <= percentile_85)

print(f"Invoices after filtering: {invoice_amounts_filtered.count():,}")

# Filtered distribution statistics
print("\n=== Filtered Distribution (85th percentile - without outliers) ===")
stats_filtered = invoice_amounts_filtered.select(
    F.min("invoice_amount").alias("min"),
    F.max("invoice_amount").alias("max"),
    F.mean("invoice_amount").alias("mean"),
    F.expr("percentile_approx(invoice_amount, 0.5)").alias("median")
)
stats_filtered.show()

# Mode for filtered data
mode_filtered = (invoice_amounts_filtered
    .groupBy("invoice_amount")
    .count()
    .orderBy(F.desc("count"))
    .limit(1)
    .select("invoice_amount")
    .first()[0]
)
print(f"Mode: £{mode_filtered}")

# Display for visualization
display(invoice_amounts_filtered)

Total invoices: 39,691

=== Full Distribution (with outliers) ===
+----+--------+-----------------+------+
| min|     max|             mean|median|
+----+--------+-----------------+------+
|0.19|168469.6|518.4292917789925|304.38|
+----+--------+-----------------+------+

Mode: £15.0


invoice_no,invoice_amount
490298,1523.2
491055,219.55
491969,6141.94
494345,374.11
495102,686.22
497027,982.26
500571,3.75
502228,188.26
509340,470.15
509348,267.83


Databricks visualization. Run in Databricks to view.


85th percentile threshold: £686.22
Invoices after filtering: 33,344

=== Filtered Distribution (85th percentile - without outliers) ===
+----+------+-----------------+------+
| min|   max|             mean|median|
+----+------+-----------------+------+
|0.19|686.22|266.5921994961612|253.42|
+----+------+-----------------+------+

Mode: £15.0


invoice_no,invoice_amount
491055,219.55
494345,374.11
495102,686.22
500571,3.75
502228,188.26
509340,470.15
509348,267.83
510990,61.2
514573,14.25
520518,50.4


Databricks visualization. Run in Databricks to view.

# Monthly Placed and Canceled Orders

In [0]:

# Section 2: Monthly Placed and Canceled Orders


from pyspark.sql import functions as F

# Exclude non-product codes (same as Section 1)
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Prepare data: filter to actual transactions, add year_month
orders_by_month = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .select("invoice_no", "invoice_date", "is_cancelled", "year_month")
    .distinct()  # One row per invoice
)

# Count placed vs canceled orders by month
monthly_orders = (orders_by_month
    .groupBy("year_month")
    .agg(
        F.sum(F.when(~F.col("is_cancelled"), 1).otherwise(0)).alias("placed_orders"),
        F.sum(F.when(F.col("is_cancelled"), 1).otherwise(0)).alias("canceled_orders")
    )
    .orderBy("year_month")
)

# Display results
print("=== Monthly Placed and Canceled Orders ===")
display(monthly_orders)

# Summary statistics
total_placed = monthly_orders.agg(F.sum("placed_orders")).first()[0]
total_canceled = monthly_orders.agg(F.sum("canceled_orders")).first()[0]

print(f"\nTotal placed orders: {total_placed:,}")
print(f"Total canceled orders: {total_canceled:,}")
print(f"Cancellation rate: {(total_canceled / total_placed * 100):.2f}%")

=== Monthly Placed and Canceled Orders ===


year_month,placed_orders,canceled_orders
200912,1923,383
201001,1287,248
201002,1721,224
201003,1937,363
201004,1566,268
201005,2000,392
201006,1838,323
201007,1657,319
201008,1598,247
201009,1972,324


Databricks visualization. Run in Databricks to view.


Total placed orders: 45,044
Total canceled orders: 7,597
Cancellation rate: 16.87%


# Monthly Sales

In [0]:
# Exclude non-product codes
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Calculate monthly sales (placed orders only)
monthly_sales = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))  # Only placed orders
    .groupBy("year_month")
    .agg(F.round(F.sum("revenue"), 2).alias("monthly_sales"))
    .withColumn("month", 
        F.to_date(
            F.concat(
                F.substring(F.col("year_month").cast("string"), 1, 4),
                F.lit("-"),
                F.substring(F.col("year_month").cast("string"), 5, 2),
                F.lit("-01")
            )
        )
    )
    .orderBy("year_month")
    .select("month", "monthly_sales")
)

print("=== Monthly Sales ===")
display(monthly_sales)

# Summary statistics
total_sales = monthly_sales.agg(F.sum("monthly_sales")).first()[0]
avg_monthly_sales = monthly_sales.agg(F.avg("monthly_sales")).first()[0]
    
print(f"\nTotal sales (all periods): £{total_sales:,.2f}")
print(f"Average monthly sales: £{avg_monthly_sales:,.2f}")

=== Monthly Sales ===


month,monthly_sales
2009-12-01,823433.37
2010-01-01,628189.4
2010-02-01,549131.15
2010-03-01,778658.51
2010-04-01,658058.9
2010-05-01,654602.17
2010-06-01,709063.12
2010-07-01,645232.88
2010-08-01,686306.0
2010-09-01,883202.96


Databricks visualization. Run in Databricks to view.


Total sales (all periods): £20,576,977.02
Average monthly sales: £823,079.08


In [0]:
# Monthly net sales (placed - canceled)
monthly_net_sales = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .groupBy("year_month")
    .agg(
        F.round(F.sum(F.when(~F.col("is_cancelled"), F.col("revenue")).otherwise(0)), 2).alias("gross_sales"),
        F.round(F.sum(F.when(F.col("is_cancelled"), F.col("revenue")).otherwise(0)), 2).alias("canceled_sales"),
        F.round(F.sum("revenue"), 2).alias("net_sales")
    )
    .withColumn("month", 
        F.to_date(
            F.concat(
                F.substring(F.col("year_month").cast("string"), 1, 4),
                F.lit("-"),
                F.substring(F.col("year_month").cast("string"), 5, 2),
                F.lit("-01")
            )
        )
    )
    .orderBy("year_month")
    .select("month", "gross_sales", "canceled_sales", "net_sales")
)

display(monthly_net_sales)

month,gross_sales,canceled_sales,net_sales
2009-12-01,823433.37,-19778.82,803654.55
2010-01-01,628189.4,-8063.84,620125.56
2010-02-01,549131.15,-12675.09,536456.06
2010-03-01,778658.51,-11052.4,767606.11
2010-04-01,658058.9,-10308.56,647750.34
2010-05-01,654602.17,-37688.38,616913.79
2010-06-01,709063.12,-26909.2,682153.92
2010-07-01,645232.88,-17410.4,627822.48
2010-08-01,686306.0,-13195.4,673110.6
2010-09-01,883202.96,-28608.23,854594.73


Databricks visualization. Run in Databricks to view.

# Monthly Sales Growth


In [0]:
# Exclude non-product codes
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Calculate monthly sales
monthly_sales_base = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))
    .groupBy("year_month")
    .agg(F.round(F.sum("revenue"), 2).alias("monthly_sales"))
    .orderBy("year_month")
)

# Create window to get previous month's sales
window_spec = Window.orderBy("year_month")

# Calculate growth percentage
monthly_sales_growth = (monthly_sales_base
    .withColumn("previous_month_sales", F.lag("monthly_sales", 1).over(window_spec))
    .withColumn("sales_growth_pct", 
        F.round(
            ((F.col("monthly_sales") - F.col("previous_month_sales")) / F.col("previous_month_sales")) * 100,
            2
        )
    )
    .withColumn("month", 
        F.to_date(
            F.concat(
                F.substring(F.col("year_month").cast("string"), 1, 4),
                F.lit("-"),
                F.substring(F.col("year_month").cast("string"), 5, 2),
                F.lit("-01")
            )
        )
    )
    .select("month", "monthly_sales", "previous_month_sales", "sales_growth_pct")
)

print("=== Monthly Sales Growth ===")
display(monthly_sales_growth)

# Summary statistics (exclude first month with null growth)
avg_growth = monthly_sales_growth.filter(F.col("sales_growth_pct").isNotNull()).agg(
    F.avg("sales_growth_pct")
).first()[0]

max_growth = monthly_sales_growth.filter(F.col("sales_growth_pct").isNotNull()).agg(
    F.max("sales_growth_pct")
).first()[0]

min_growth = monthly_sales_growth.filter(F.col("sales_growth_pct").isNotNull()).agg(
    F.min("sales_growth_pct")
).first()[0]

print(f"\nAverage monthly growth: {avg_growth:.2f}%")
print(f"Highest growth: {max_growth:.2f}%")
print(f"Lowest growth: {min_growth:.2f}%")

=== Monthly Sales Growth ===


/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


month,monthly_sales,previous_month_sales,sales_growth_pct
2009-12-01,823433.37,null,null
2010-01-01,628189.4,823433.37,-23.71
2010-02-01,549131.15,628189.4,-12.59
2010-03-01,778658.51,549131.15,41.8
2010-04-01,658058.9,778658.51,-15.49
2010-05-01,654602.17,658058.9,-0.53
2010-06-01,709063.12,654602.17,8.32
2010-07-01,645232.88,709063.12,-9.0
2010-08-01,686306.0,645232.88,6.37
2010-09-01,883202.96,686306.0,28.69


Databricks visualization. Run in Databricks to view.


Average monthly growth: 3.09%
Highest growth: 45.03%
Lowest growth: -57.62%


# Monthly Active Users

In [0]:
# Exclude non-product codes
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Calculate monthly active users (unique customers per month)
monthly_active_users = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))  # Only completed purchases
    .filter(F.col("customer_id").isNotNull())  # Must have customer_id
    .groupBy("year_month")
    .agg(F.countDistinct("customer_id").alias("active_users"))
    .withColumn("month", 
        F.to_date(
            F.concat(
                F.substring(F.col("year_month").cast("string"), 1, 4),
                F.lit("-"),
                F.substring(F.col("year_month").cast("string"), 5, 2),
                F.lit("-01")
            )
        )
    )
    .orderBy("year_month")
    .select("month", "active_users")
)

print("=== Monthly Active Users ===")
display(monthly_active_users)

# Summary statistics
total_unique_customers = df.filter(
    ~F.col("stock_code").isin(exclude_codes) &
    ~F.col("is_cancelled") &
    F.col("customer_id").isNotNull()
).select("customer_id").distinct().count()

avg_monthly_users = monthly_active_users.agg(F.avg("active_users")).first()[0]
max_monthly_users = monthly_active_users.agg(F.max("active_users")).first()[0]
min_monthly_users = monthly_active_users.agg(F.min("active_users")).first()[0]

print(f"\nTotal unique customers (all time): {total_unique_customers:,}")
print(f"Average monthly active users: {avg_monthly_users:,.0f}")
print(f"Highest monthly active users: {max_monthly_users:,}")
print(f"Lowest monthly active users: {min_monthly_users:,}")

=== Monthly Active Users ===


month,active_users
2009-12-01,954
2010-01-01,702
2010-02-01,773
2010-03-01,1053
2010-04-01,939
2010-05-01,966
2010-06-01,1036
2010-07-01,925
2010-08-01,910
2010-09-01,1138


Databricks visualization. Run in Databricks to view.


Total unique customers (all time): 5,857
Average monthly active users: 1,021
Highest monthly active users: 1,662
Lowest monthly active users: 614


# New and Existing Users



In [0]:
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Find each customer's first purchase month
first_purchase = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(F.min("year_month").alias("first_purchase_month"))
)

# Join back to transactions to classify new vs existing
user_classification = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))
    .filter(F.col("customer_id").isNotNull())
    .join(first_purchase, "customer_id")
    .withColumn("user_type", 
        F.when(F.col("year_month") == F.col("first_purchase_month"), "new")
        .otherwise("existing")
    )
)

# Count new vs existing users per month
new_existing_monthly = (user_classification
    .groupBy("year_month", "user_type")
    .agg(F.countDistinct("customer_id").alias("user_count"))
    .groupBy("year_month")
    .pivot("user_type")
    .sum("user_count")
    .na.fill(0)
    .withColumn("month", 
        F.to_date(F.concat(
            F.substring(F.col("year_month").cast("string"), 1, 4),
            F.lit("-"),
            F.substring(F.col("year_month").cast("string"), 5, 2),
            F.lit("-01")
        ))
    )
    .orderBy("year_month")
    .select("month", "new", "existing")
)

print("=== New vs Existing Users by Month ===")
display(new_existing_monthly)

=== New vs Existing Users by Month ===


month,new,existing
2009-12-01,954,0
2010-01-01,368,334
2010-02-01,377,396
2010-03-01,440,613
2010-04-01,294,645
2010-05-01,255,711
2010-06-01,267,769
2010-07-01,186,739
2010-08-01,163,747
2010-09-01,239,899


Databricks visualization. Run in Databricks to view.

In [0]:
exclude_codes = ['M', 'B', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'D', 'CRUK']

# Reference date (use max date in dataset)
max_date = df.select(F.max("invoice_date")).first()[0]
print(f"Reference date: {max_date}")

# Calculate RFM metrics per customer
rfm = (df
    .filter(~F.col("stock_code").isin(exclude_codes))
    .filter(~F.col("is_cancelled"))
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .groupBy("customer_id")
    .agg(
        F.datediff(F.lit(max_date), F.max("invoice_date")).alias("recency"),  # Days since last purchase
        F.countDistinct("invoice_no").alias("frequency"),  # Number of orders
        F.round(F.sum("revenue"), 2).alias("monetary")  # Total spend
    )
)

print("=== RFM Table ===")
display(rfm)

# RFM summary statistics
print("\n=== RFM Statistics ===")
rfm_stats = rfm.select(
    F.min("recency").alias("min_recency"),
    F.max("recency").alias("max_recency"),
    F.avg("recency").alias("avg_recency"),
    F.min("frequency").alias("min_frequency"),
    F.max("frequency").alias("max_frequency"),
    F.avg("frequency").alias("avg_frequency"),
    F.min("monetary").alias("min_monetary"),
    F.max("monetary").alias("max_monetary"),
    F.avg("monetary").alias("avg_monetary")
)
display(rfm_stats)


Reference date: 2011-12-09 12:50:00
=== RFM Table ===


customer_id,recency,frequency,monetary
14640,24,10,3667.06
16211,88,3,1090.59
15392,4,7,2911.62
15961,379,1,451.29
17557,14,4,575.46
13995,15,14,3902.01
18008,70,23,19787.13
13126,4,13,3998.25
17518,425,2,329.42
15502,15,12,10532.52



=== RFM Statistics ===


min_recency,max_recency,avg_recency,min_frequency,max_frequency,avg_frequency,min_monetary,max_monetary,avg_monetary
0,738,199.8266142808336,1,375,6.273317389818927,2.95,608821.65,3004.2171472497425


# RFM Segmentation

In [0]:
# Create RFM scores (quintiles: 1-5)
rfm_with_scores = (rfm
    .withColumn("r_score", 
        F.when(F.col("recency") <= rfm.approxQuantile("recency", [0.2], 0.01)[0], 5)
        .when(F.col("recency") <= rfm.approxQuantile("recency", [0.4], 0.01)[0], 4)
        .when(F.col("recency") <= rfm.approxQuantile("recency", [0.6], 0.01)[0], 3)
        .when(F.col("recency") <= rfm.approxQuantile("recency", [0.8], 0.01)[0], 2)
        .otherwise(1)
    )
    .withColumn("f_score", 
        F.when(F.col("frequency") >= rfm.approxQuantile("frequency", [0.8], 0.01)[0], 5)
        .when(F.col("frequency") >= rfm.approxQuantile("frequency", [0.6], 0.01)[0], 4)
        .when(F.col("frequency") >= rfm.approxQuantile("frequency", [0.4], 0.01)[0], 3)
        .when(F.col("frequency") >= rfm.approxQuantile("frequency", [0.2], 0.01)[0], 2)
        .otherwise(1)
    )
    .withColumn("m_score", 
        F.when(F.col("monetary") >= rfm.approxQuantile("monetary", [0.8], 0.01)[0], 5)
        .when(F.col("monetary") >= rfm.approxQuantile("monetary", [0.6], 0.01)[0], 4)
        .when(F.col("monetary") >= rfm.approxQuantile("monetary", [0.4], 0.01)[0], 3)
        .when(F.col("monetary") >= rfm.approxQuantile("monetary", [0.2], 0.01)[0], 2)
        .otherwise(1)
    )
    .withColumn("rfm_score", F.concat(F.col("r_score"), F.col("f_score"), F.col("m_score")))
)

# Customer segmentation based on RFM
rfm_segments = (rfm_with_scores
    .withColumn("segment",
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
        .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3) & (F.col("m_score") >= 3), "Loyal Customers")
        .when((F.col("r_score") >= 4) & (F.col("f_score") <= 2), "New Customers")
        .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), "At Risk")
        .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2), "Lost")
        .otherwise("Potential Loyalists")
    )
)

print("\n=== RFM Segmentation ===")
display(rfm_segments)

# Segment distribution
segment_counts = (rfm_segments
    .groupBy("segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("monetary"), 2).alias("avg_monetary")
    )
    .orderBy(F.desc("customer_count"))
)

print("\n=== Customer Segments ===")
display(segment_counts)


=== RFM Segmentation ===


customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
14640,24,10,3667.06,4,5,5,455,Champions
16211,88,3,1090.59,3,3,3,333,Loyal Customers
15392,4,7,2911.62,5,4,5,545,Champions
15961,379,1,451.29,2,2,2,222,Lost
17557,14,4,575.46,5,4,2,542,Potential Loyalists
13995,15,14,3902.01,5,5,5,555,Champions
18008,70,23,19787.13,3,5,5,355,Loyal Customers
13126,4,13,3998.25,5,5,5,555,Champions
17518,425,2,329.42,1,3,2,132,At Risk
15502,15,12,10532.52,5,5,5,555,Champions



=== Customer Segments ===


segment,customer_count,avg_monetary
Champions,1368,8931.97
At Risk,1252,1602.02
Loyal Customers,1189,2210.25
Lost,1143,331.07
Potential Loyalists,659,407.83
New Customers,243,357.46
